In [1]:
import jax
import jax.numpy as jnp
import qcsys as qs
import scqubits
import numpy as np
import qutip

jax.config.update('jax_platform_name', 'cpu')
# jax.config.update('jax_enable_x64', True)
def transform_op_into_dressed_basis_jax(op_matrix, dressed_evecs):
    """
    Transform an operator into the dressed basis using JAX.

    Parameters:
    - op_matrix: A 2D JAX array representing the operator's matrix.
    - dressed_evecs: A 2D JAX array representing the dressed eigenvectors.

    Returns:
    - A 2D JAX array representing the transformed operator.
    """
    S = jnp.conj(dressed_evecs)
    data = jnp.dot(S, jnp.dot(op_matrix, S.T.conj()))
    return data

# fluxonium

In [2]:

qsf = qs.Fluxonium.create(
    25,
    {"Ej": 2.7, "Ec": 0.6, "El": 0.13, "phi_ext": 0.0},
    N_pre_diag=100,
    use_linear = False
)

scf = scqubits.Fluxonium(EJ=2.7,
                        EC=0.6,
                        EL=0.13,
                        flux=0,cutoff=100,
                        truncated_dim=25)


np.array_equal(np.array(scf.n_operator())[:25,:25] , np.array(qsf.ops['n'].data,dtype = 'complex128')), \
    np.allclose(np.array(qutip.Qobj(scf.hamiltonian()).tidyup()), np.array(qsf.get_H_full().data,dtype = 'complex128'),atol=1e-08)


(False, True)

In [3]:
qutip.Qobj(np.array(qsf.ops['n'].data)).tidyup()

Quantum object: dims = [[25], [25]], shape = (25, 25), type = oper, isherm = True
Qobj data =
[[0.+0.00000000e+00j 0.-1.26767473e-01j 0.+0.00000000e+00j
  0.+5.32775444e-01j 0.+0.00000000e+00j 0.+9.50553963e-02j
  0.+0.00000000e+00j 0.+6.47689721e-02j 0.+0.00000000e+00j
  0.-2.38813613e-02j 0.+0.00000000e+00j 0.+1.83069255e-02j
  0.+0.00000000e+00j 0.-1.22788330e-02j 0.+0.00000000e+00j
  0.-7.45726981e-03j 0.+0.00000000e+00j 0.+4.51693382e-03j
  0.+0.00000000e+00j 0.+2.54011597e-03j 0.+0.00000000e+00j
  0.+1.41769958e-03j 0.+0.00000000e+00j 0.+7.67444720e-04j
  0.+0.00000000e+00j]
 [0.+1.26767473e-01j 0.+0.00000000e+00j 0.+2.25618709e-02j
  0.+0.00000000e+00j 0.+3.03754709e-01j 0.+0.00000000e+00j
  0.+4.01604395e-01j 0.+0.00000000e+00j 0.+2.33167582e-02j
  0.+0.00000000e+00j 0.+2.60312483e-02j 0.+0.00000000e+00j
  0.-4.01725281e-02j 0.+0.00000000e+00j 0.-2.37697788e-03j
  0.+0.00000000e+00j 0.+1.60480681e-02j 0.+0.00000000e+00j
  0.-1.04790654e-02j 0.+0.00000000e+00j 0.+2.20870538e-03j

In [4]:
qutip.Qobj((np.array(scf.n_operator())[:25,:25]))

Quantum object: dims = [[25], [25]], shape = (25, 25), type = oper, isherm = True
Qobj data =
[[0.+0.j         0.-0.28685375j 0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j        ]
 [0.+0.28685375j 0.+0.j         0.-0.40567246j 0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j        ]
 [0.+0.j         0.+0.40567246j 0.+0.j         0.-0.49684527j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.j
  0.+0.j         0.+0.j         0.+0.j         0.+0.

# transmon

In [6]:
qst_sc = qs.SingleChargeTransmon.create(
    N = 10,
    params = {"Ej": 40, "Ec": 0.5,"ng":0.0},
    N_max_charge=30
)

sct = scqubits.Transmon(
    EJ=40,
    EC=0.5,
    ng=0.0,
    ncut=30,
    truncated_dim = 10
    )

np.array_equal(np.array(sct.n_operator()),np.array(qst_sc.linear_ops['n'].data)), \
    np.array_equal(np.array(sct.hamiltonian()),np.array(qst_sc.get_H_full().data))

(True, True)

# hilbertspace: Make bare hamiltonian the same 

In [7]:
import jaxquantum as jqt
devices = [qsf, qst_sc]
f_indx = 0
t_indx = 1
Ns = [device.N for device in devices]
tn = qs.promote(qst_sc.common_ops()["n"], t_indx, Ns)
fn = qs.promote(qsf.ops["n"], f_indx, Ns)
g_tf = 0.2
system = qs.System.create(devices, couplings=[
    # g_tf * tn @ fn
])
system.params["g_tf"] = g_tf
Es, kets = system.calculate_eig_linear()


hilbertspace = scqubits.HilbertSpace([scf, sct])
hilbertspace.add_interaction(g_strength=g_tf,op1=scf.n_operator, op2=sct.n_operator,add_hc=False)
evals, evecs = hilbertspace.hamiltonian().eigenstates()


# Tmon part bare H
evals = sct.eigenvals(evals_count=sct.truncated_dim)
scq_tmon_bare = np.array(hilbertspace.diag_hamiltonian(sct, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_tmon_bare = np.array(system.promote(qst_sc.get_H(), 1).data)

# Fluxonium part bare H
evals = scf.eigenvals(evals_count=scf.truncated_dim)
scq_f_bare = np.array(hilbertspace.diag_hamiltonian(scf, evals).full())

I_ops = [jqt.identity(N) for N in system.Ns]
qcs_f_bare = np.array(system.promote(qsf.get_H(), 0).data)


np.allclose(scq_f_bare,qcs_f_bare), \
    np.allclose(scq_tmon_bare,qcs_tmon_bare), \
        np.allclose(np.array(system.get_H_bare().data), np.array(hilbertspace.bare_hamiltonian()))

(True, True, True)

# hilbertspace: Make interaction hamiltonian the same

In [5]:
system = qs.System.create(devices, couplings=[
    g_tf * tn @ fn
])

TypeError: incompatible dimensions [[25, 61], [25, 61]] and [[25, 4], [25, 4]]

In [ ]:
tn @ fn

TypeError: incompatible dimensions [[25, 61], [25, 61]] and [[25, 4], [25, 4]]

scqubits interaction hamiltonian is a sum over interaction terms
    interaction terms = g * id_wrapped_op1 * id_wrapped_op2

where id_wrapped_op = spec_utils.identity_wrap(
                    operator,
                    subsystem_list[subsys_index],
                    subsystem_list,
                    evecs=evecs,
                    op_in_eigenbasis=op_in_eigenbasis,
                )

basically spec_utils.identity_wrap tensors the convert_operator_to_qobj() transformed operator with identities

convert_operator_to_qobj() basically converts the operator into dressed eigenbasis

In [9]:
from scqubits.utils.spectrum_utils import order_eigensystem
import scipy as sp
operator = sct.n_operator()
subsystem = hilbertspace.subsystem_list[1]

evals_count = 10
hamiltonian_mat = subsystem.hamiltonian()
evals, evecs = sp.linalg.eigh(
    hamiltonian_mat,
    eigvals_only=False,
    subset_by_index=(0, evals_count - 1),
    check_finite=False,
)
evals, evecs = order_eigensystem(evals, evecs)
state_table = evecs
states_in_columns = state_table
mtable = states_in_columns.conj().T @ operator @ states_in_columns
subsys_operator =  qutip.Qobj(mtable)


In [10]:
subsys_operator.tidyup()

Quantum object: dims = [[10], [10]], shape = (10, 10), type = oper, isherm = True
Qobj data =
[[ 0.00000000e+00 -1.23104130e+00  0.00000000e+00  3.39734687e-02
   0.00000000e+00 -2.00372233e-03  0.00000000e+00 -2.13303579e-04
   0.00000000e+00  3.00207452e-05]
 [-1.23104130e+00  0.00000000e+00 -1.70023304e+00  0.00000000e+00
  -7.25831861e-02  0.00000000e+00  5.84820901e-03  0.00000000e+00
   1.05717494e-03  0.00000000e+00]
 [ 0.00000000e+00 -1.70023304e+00  0.00000000e+00  2.02624923e+00
   0.00000000e+00 -1.23877796e-01  0.00000000e+00 -1.31891806e-02
   0.00000000e+00  1.85627353e-03]
 [ 3.39734687e-02  0.00000000e+00  2.02624923e+00  0.00000000e+00
   2.26481778e+00  0.00000000e+00 -1.92218298e-01  0.00000000e+00
  -3.47619610e-02  0.00000000e+00]
 [ 0.00000000e+00 -7.25831861e-02  0.00000000e+00  2.26481778e+00
   0.00000000e+00 -2.43026117e+00  0.00000000e+00 -2.77878186e-01
   0.00000000e+00  3.91468648e-02]
 [-2.00372233e-03  0.00000000e+00 -1.23877796e-01  0.00000000e+00
  -2.

In [11]:
qutip.Qobj(np.array(qst_sc.get_op_in_H_eigenbasis(qst_sc.linear_ops['n']).data)).tidyup()


Quantum object: dims = [[10], [10]], shape = (10, 10), type = oper, isherm = True
Qobj data =
[[ 0.00000000e+00  1.23104130e+00  0.00000000e+00 -3.39734687e-02
   0.00000000e+00  2.00372233e-03  0.00000000e+00 -2.13303579e-04
   0.00000000e+00  3.00207452e-05]
 [ 1.23104130e+00  0.00000000e+00 -1.70023304e+00  0.00000000e+00
   7.25831861e-02  0.00000000e+00 -5.84820901e-03  0.00000000e+00
   1.05717494e-03  0.00000000e+00]
 [ 0.00000000e+00 -1.70023304e+00  0.00000000e+00  2.02624923e+00
   0.00000000e+00 -1.23877796e-01  0.00000000e+00  1.31891806e-02
   0.00000000e+00 -1.85627353e-03]
 [-3.39734687e-02  0.00000000e+00  2.02624923e+00  0.00000000e+00
  -2.26481778e+00  0.00000000e+00  1.92218298e-01  0.00000000e+00
  -3.47619610e-02  0.00000000e+00]
 [ 0.00000000e+00  7.25831861e-02  0.00000000e+00 -2.26481778e+00
   0.00000000e+00  2.43026117e+00  0.00000000e+00 -2.77878186e-01
   0.00000000e+00  3.91468648e-02]
 [ 2.00372233e-03  0.00000000e+00 -1.23877796e-01  0.00000000e+00
   2.

In [19]:
import jax.numpy as jnp
import numpy as np
import scipy as sp

data = qst_sc._get_H_in_linear_basis().data

evecs_list = []
for func in [jnp.linalg.eig, np.linalg.eig, sp.linalg.eig, 
             jnp.linalg.eigh, np.linalg.eigh, sp.linalg.eigh]:
    
    evals, evecs = func(data) 
    idxs_sorted = jnp.argsort(evals)
    evecs_list.append(evecs[:, idxs_sorted])


In [24]:
# 1 the eig function of the three packages returns the same value
np.sum(np.abs(evecs_list[0]-evecs_list[1])>1e-6), \
      np.sum(np.abs(evecs_list[0]-evecs_list[2])>1e-6)

(0, 0)

In [29]:
# 2 the eigh function of the three packages are not the same, jax.numpy and numpy are more similar
np.sum(np.abs(evecs_list[3])-np.abs(evecs_list[4])>1e-1), \
      np.sum(np.abs(evecs_list[3])-np.abs(evecs_list[5])>1e-1), \
      np.sum(np.abs(evecs_list[4])-np.abs(evecs_list[5])>1e-1)

(12, 64, 76)